In [ ]:
# Importing libraries and defining paths

from pathlib import Path
import shutil, yaml

EXPORT = Path("../datasets/raw_export")
COLLAPSED = Path("../datasets/collapsed")

In [ ]:
# Inspecting labels

for split in ["train", "valid"]:

    total_boxes = 0
    empty_files = 0
    class_ids = set()

    files = list((EXPORT / split / "labels").glob("*.txt"))

    for f in files:
        text = f.read_text().strip()

        if text == "":
            empty_files += 1
            continue

        lines = text.split("\n")
        total_boxes += len(lines)

        for line in lines:
            class_ids.add(line.split()[0])

    print(f"{split.capitalize()} Label Files: {len(files)}")
    print(f"{split.capitalize()} Empty Files: {empty_files}")
    print(f"{split.capitalize()} Boxes: {total_boxes}")
    print(f"{split.capitalize()} Classes: {class_ids}")

In [ ]:
# Collapsing into a single class

for split in ["train", "valid"]:
    for folder in ["images", "labels"]:
        src = EXPORT / split / folder
        dst = COLLAPSED / split / folder
        dst.mkdir(parents=True, exist_ok=True)

        for f in src.glob("*"):
            if folder == "images":
                shutil.copy2(f, dst / f.name)
            elif folder == "labels":
                text = f.read_text().strip()
                
                if text == "":
                    (dst / f.name).write_text("")
                    continue
        
                new_lines = []
                for line in text.split("\n"):
                    parts = line.split()
                    parts[0] = "0"
                    new_lines.append(" ".join(parts))

                (dst / f.name).write_text("\n".join(new_lines))

config = {
    "path": str(COLLAPSED.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "nc": 1,
    "names": ["object"],
}

(COLLAPSED / "data.yaml").write_text(yaml.safe_dump(config))
print((COLLAPSED / "data.yaml").read_text())

In [ ]:
# Confirming collapse

for split in ["train", "valid"]:

    total_boxes = 0
    empty_files = 0
    class_ids = set()

    files = list((COLLAPSED / split / "labels").glob("*.txt"))

    for f in files:
        text = f.read_text().strip()

        if text == "":
            empty_files += 1
            continue

        lines = text.split("\n")
        total_boxes += len(lines)

        for line in lines:
            class_ids.add(line.split()[0])

    print(f"{split.capitalize()} Label Files: {len(files)}")
    print(f"{split.capitalize()} Empty Files: {empty_files}")
    print(f"{split.capitalize()} Boxes: {total_boxes}")
    print(f"{split.capitalize()} Classes: {class_ids}")